# Basic RAG Pipeline Implementation
In this notebook, we will build a standard RAG pipeline from scratch: Loading a document, splitting it, creating embeddings, storing them in ChromaDB, and retrieving the answers.


In [ ]:
# 1. Imports
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


In [ ]:
# 2. Load and Split the Document
# Simulate loading a file (we'll just create a dummy text file for this example)
with open("dummy_data.txt", "w") as f:
    f.write("The company refund policy allows returns within 30 days. Software purchases are non-refundable.")

loader = TextLoader("dummy_data.txt")
docs = loader.load()

# Split into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
chunks = text_splitter.split_documents(docs)
print(f"Split into {len(chunks)} chunks.")


In [ ]:
# 3. Embeddings & Vector Store
# We convert chunks to numbers and store them in ChromaDB.
embeddings = OpenAIEmbeddings()

# --- METHOD 1: IN-MEMORY (RAM) ---
# Good for training and practice. Deletes automatically when script finishes.
vectorstore_ram = Chroma.from_documents(documents=chunks, embedding=embeddings)

# --- METHOD 2: ON-DISK (PERMANENT STORAGE) ---
# Good for production. Creates a physical folder so you don't have to re-embed data next time.
# vectorstore_disk = Chroma.from_documents(
#     documents=chunks, 
#     embedding=embeddings, 
#     persist_directory="./chroma_database"
# )

# Turn the vector store into a Retriever (We will use the RAM version for this practice)
retriever = vectorstore_ram.as_retriever(search_kwargs={"k": 1}) # Fetch top 1 result


In [ ]:
# 4. Build the RAG Chain
llm = ChatOpenAI(temperature=0)

template = """Answer the question based ONLY on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

# LCEL Chain: Fetch context -> Inject into Prompt -> LLM -> Output String
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()} 
    | prompt 
    | llm 
    | StrOutputParser()
)


In [ ]:
# 5. Execute
response = rag_chain.invoke("What is the refund policy for software?")
print(response)
